In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('creditcard.csv')
print("Shape:", df.shape)
print(df.head())
print(df.info())
print(df.isnull().sum())
print("\nFraud vs Non-Fraud:")
print(df['Class'].value_counts())

Shape: (284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26 

In [3]:
# Check fraud rate
fraud_rate = df['Class'].mean() * 100
print(f"Overall Fraud Rate: {fraud_rate:.3f}%")

# Separate fraud and non-fraud
fraud = df[df['Class'] == 1]
legit = df[df['Class'] == 0]
print(f"Fraudulent transactions: {len(fraud)}")
print(f"Legitimate transactions: {len(legit)}")

# Add transaction amount groups
df['Amount_Group'] = pd.cut(df['Amount'],
    bins=[0, 50, 200, 500, 2000, 30000],
    labels=['Very Low <$50', 'Low $50-200', 'Medium $200-500',
            'High $500-2K', 'Very High >$2K'])

# Add time of day (dataset records seconds — convert to hour)
df['Hour'] = (df['Time'] / 3600 % 24).astype(int)
df['Time_Group'] = pd.cut(df['Hour'],
    bins=[0, 6, 12, 18, 24],
    labels=['Night 12-6am', 'Morning 6-12pm',
            'Afternoon 12-6pm', 'Evening 6-12pm'])

print("✅ Done!")

Overall Fraud Rate: 0.173%
Fraudulent transactions: 492
Legitimate transactions: 284315
✅ Done!


In [5]:
# 1. Fraud rate by amount group
print("Fraud Rate by Transaction Amount:")
print(df.groupby('Amount_Group', observed=True)['Class']
      .mean().mul(100).round(3))

# 2. Fraud rate by time of day
print("\nFraud Rate by Time of Day:")
print(df.groupby('Time_Group', observed=True)['Class']
      .mean().mul(100).round(3))

# 3. Average fraudulent amount vs legitimate
print(f"\nAvg Fraudulent Transaction: ${fraud['Amount'].mean():.2f}")
print(f"Avg Legitimate Transaction: ${legit['Amount'].mean():.2f}")

# 4. Top transaction amounts in fraud
print(f"\nMax Fraudulent Amount: ${fraud['Amount'].max():.2f}")
print(f"Most common fraud amount range: {fraud['Amount'].describe()}")

Fraud Rate by Transaction Amount:
Amount_Group
Very Low <$50      0.147
Low $50-200        0.156
Medium $200-500    0.254
High $500-2K       0.402
Very High >$2K     0.148
Name: Class, dtype: float64

Fraud Rate by Time of Day:
Time_Group
Night 12-6am        0.624
Morning 6-12pm      0.153
Afternoon 12-6pm    0.153
Evening 6-12pm      0.109
Name: Class, dtype: float64

Avg Fraudulent Transaction: $122.21
Avg Legitimate Transaction: $88.29

Max Fraudulent Amount: $2125.87
Most common fraud amount range: count     492.000000
mean      122.211321
std       256.683288
min         0.000000
25%         1.000000
50%         9.250000
75%       105.890000
max      2125.870000
Name: Amount, dtype: float64


In [9]:
# Keep ALL rows but only selected columns
df_tableau = df[['Amount', 'Hour', 'Class', 'Amount_Group', 'Time_Group']].copy()
df_tableau.columns = ['Amount', 'Hour', 'Fraud_Flag', 'Amount_Group', 'Time_Group']

df_tableau.to_csv('fraud_cleaned.csv', index=False)
print(f"✅ Saved! Total rows: {len(df_tableau)}")
print(f"Fraud cases: {df_tableau['Fraud_Flag'].sum()}")
print(f"Fraud rate: {df_tableau['Fraud_Flag'].mean()*100:.3f}%")

✅ Saved! Total rows: 284807
Fraud cases: 492
Fraud rate: 0.173%
